In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from keras import layers, Model
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
import warnings
warnings.filterwarnings('ignore')

In [ ]:
class EnhancedGRANXModel(tf.keras.Model):
    def __init__(self, sequence_length, n_features, weather_features=5, tariff_features=3, 
                 hidden_units=128, attention_heads=8, correction_rate=0.01, dropout_rate=0.2):
        super(EnhancedGRANXModel, self).__init__()
        
        self.sequence_length = sequence_length
        self.n_features = n_features
        self.weather_features = weather_features
        self.tariff_features = tariff_features
        self.hidden_units = hidden_units
        self.attention_heads = attention_heads
        self.correction_rate = correction_rate
        
        # Input fusion layers for heterogeneous data
        self.meter_projection = layers.Dense(hidden_units, activation='relu', name='meter_proj')
        self.weather_projection = layers.Dense(hidden_units//2, activation='relu', name='weather_proj')
        self.tariff_projection = layers.Dense(hidden_units//4, activation='relu', name='tariff_proj')
        self.fusion_projection = layers.Dense(hidden_units, activation='relu', name='fusion_proj')
        
        # Spatial pattern extraction (CNN-inspired)
        self.spatial_conv = [
            layers.Conv1D(64, 3, padding='same', activation='relu'),
            layers.Conv1D(128, 3, padding='same', activation='relu'),
            layers.BatchNormalization(),
            layers.Dropout(dropout_rate)
        ]
        
        # Temporal modeling (GRU/LSTM hybrid)
        self.temporal_gru = layers.GRU(hidden_units, return_sequences=True, dropout=dropout_rate)
        self.temporal_lstm = layers.LSTM(hidden_units//2, return_sequences=True, dropout=dropout_rate)
        self.temporal_projector = layers.Dense(hidden_units, activation='relu')

        
        # Multi-scale attention
        self.feature_attention = layers.MultiHeadAttention(
            num_heads=self.attention_heads,
            key_dim=self.hidden_units // self.attention_heads
        )
        self.temporal_attention = layers.MultiHeadAttention(
            num_heads=self.attention_heads // 2,
            key_dim=(self.hidden_units // 2) // (self.attention_heads // 2)
        )

        # Feature importance (fixed version)
        self.feature_projector = layers.Dense(hidden_units, activation='relu')
        self.importance_scorer = layers.Dense(hidden_units, activation='sigmoid')
        self.gradient_correction = layers.Dense(hidden_units, activation='tanh')
        
        # Adaptive fusion layer
        self.fusion_weights = layers.Dense(3, activation='softmax', name='fusion_weights')
        self.fusion_dense = layers.Dense(hidden_units, activation='relu')
        
        # Output layers
        self.global_pool = layers.GlobalAveragePooling1D()
        self.output_dense = layers.Dense(hidden_units//2, activation='relu')
        self.final_output = layers.Dense(1, activation='linear')
        
        # Regularization
        self.dropout = layers.Dropout(dropout_rate)
        self.layer_norm = layers.LayerNormalization()

    def fuse_heterogeneous_inputs(self, meter_data, weather_data, tariff_data):
        meter_proj = self.meter_projection(meter_data)
        weather_proj = self.weather_projection(weather_data)
        tariff_proj = self.tariff_projection(tariff_data)
        
        weather_broadcast = tf.tile(tf.expand_dims(weather_proj, 1), [1, self.sequence_length, 1])
        tariff_broadcast = tf.tile(tf.expand_dims(tariff_proj, 1), [1, self.sequence_length, 1])
        
        fused_features = tf.concat([meter_proj, weather_broadcast, tariff_broadcast], axis=-1)
        fused_features = self.fusion_projection(fused_features)
        
        return fused_features

    def extract_spatial_patterns(self, x):
        for layer in self.spatial_conv:
            x = layer(x)
        return x

    def model_temporal_dependencies(self, x):
        gru_out = self.temporal_gru(x)       # [batch, seq_len, 128]
        lstm_out = self.temporal_lstm(x)     # [batch, seq_len, 64]
        combined = tf.concat([gru_out, lstm_out], axis=-1)  # [batch, seq_len, 192]
        projected = self.temporal_projector(combined)       # [batch, seq_len, 128]
        return projected


    def apply_dynamic_attention(self, x):
        feature_attended = self.feature_attention(x, x)
        temporal_attended = self.temporal_attention(feature_attended, feature_attended)
        return feature_attended, temporal_attended

    def compute_feature_importance(self, x, training=None):
        x_proj = self.feature_projector(x)
        importance_scores = self.importance_scorer(x_proj)
        weighted_features = x_proj * importance_scores
        
        if training:
            correction = self.gradient_correction(weighted_features)
            corrected_features = weighted_features - self.correction_rate * correction
        else:
            corrected_features = weighted_features

        return corrected_features, importance_scores

    def call(self, inputs, training=None):
        meter_data, weather_data, tariff_data = inputs
        
        fused_features = self.fuse_heterogeneous_inputs(meter_data, weather_data, tariff_data)
        spatial_features = self.extract_spatial_patterns(fused_features)
        temporal_features = self.model_temporal_dependencies(spatial_features)
        feature_attended, temporal_attended = self.apply_dynamic_attention(temporal_features)
        corrected_features, importance_scores = self.compute_feature_importance(
            temporal_attended, training=training
        )
        
        fusion_weights = self.fusion_weights(self.global_pool(corrected_features))
        
        pathway1 = self.global_pool(feature_attended)
        pathway2 = self.global_pool(temporal_attended)  
        pathway3 = self.global_pool(corrected_features)
        
        fused_output = (fusion_weights[:, 0:1] * pathway1 + 
                        fusion_weights[:, 1:2] * pathway2 + 
                        fusion_weights[:, 2:3] * pathway3)
        
        fused_output = self.layer_norm(fused_output)
        fused_output = self.dropout(fused_output, training=training)
        output_features = self.output_dense(fused_output)
        return self.final_output(output_features)


In [ ]:
class HeterogeneousDataProcessor:
    def __init__(self, sequence_length=24):
        self.sequence_length = sequence_length
        self.meter_scaler = StandardScaler()
        self.weather_scaler = StandardScaler()
        self.tariff_scaler = StandardScaler()
        self.target_scaler = MinMaxScaler()
    
    def create_heterogeneous_synthetic_data(self, n_samples=2000):
        np.random.seed(42)
        
        hours = np.arange(n_samples) % 24
        days = np.arange(n_samples) // 24
        months = (np.arange(n_samples) // (24 * 30)) % 12
        
        # Smart meter data
        base_consumption = (
            400 + 300 * np.sin(2 * np.pi * hours / 24) +
            100 * np.sin(2 * np.pi * days / 7) +
            50 * np.sin(2 * np.pi * months / 12) +
            np.random.normal(0, 30, n_samples)
        )
        
        meter_features = {
            'powerallphases': base_consumption,
            'powerl1': base_consumption * 0.4 + np.random.normal(0, 15, n_samples),
            'powerl2': base_consumption * 0.3 + np.random.normal(0, 10, n_samples),
            'powerl3': base_consumption * 0.3 + np.random.normal(0, 10, n_samples),
            'voltagel1': 230 + np.random.normal(0, 5, n_samples),
            'voltagel2': 230 + np.random.normal(0, 5, n_samples),
            'currentl1': base_consumption * 0.01 + np.random.normal(0, 1, n_samples),
        }
        
        # Weather data (daily patterns)
        temp_base = 20 + 10 * np.sin(2 * np.pi * months / 12)
        weather_features = {
            'temperature': temp_base + 5 * np.sin(2 * np.pi * hours / 24) + np.random.normal(0, 2, n_samples),
            'humidity': 60 + 20 * np.sin(2 * np.pi * (hours + 6) / 24) + np.random.normal(0, 5, n_samples),
            'wind_speed': 5 + 3 * np.random.exponential(1, n_samples),
            'solar_irradiance': np.maximum(0, 800 * np.sin(np.pi * (hours - 6) / 12) + np.random.normal(0, 50, n_samples)),
            'pressure': 1013 + np.random.normal(0, 10, n_samples)
        }
        
        # Tariff data (time-of-use pricing)
        peak_hours = ((hours >= 17) & (hours <= 21)).astype(float)
        off_peak = ((hours >= 22) | (hours <= 6)).astype(float)
        
        tariff_features = {
            'electricity_rate': 0.12 + 0.08 * peak_hours - 0.04 * off_peak + np.random.normal(0, 0.01, n_samples),
            'demand_charge': 0.05 + 0.03 * peak_hours + np.random.normal(0, 0.005, n_samples),
            'grid_load_factor': 0.7 + 0.2 * peak_hours + np.random.normal(0, 0.05, n_samples)
        }
        
        return pd.DataFrame(meter_features), pd.DataFrame(weather_features), pd.DataFrame(tariff_features)
    
    def create_sequences_heterogeneous(self, meter_data, weather_data, tariff_data, target_col='powerallphases'):
        meter_scaled = self.meter_scaler.fit_transform(meter_data)
        weather_scaled = self.weather_scaler.fit_transform(weather_data)
        tariff_scaled = self.tariff_scaler.fit_transform(tariff_data)
        
        target = meter_data[target_col].values
        target_scaled = self.target_scaler.fit_transform(target.reshape(-1, 1)).flatten()
        
        X_meter, X_weather, X_tariff, y = [], [], [], []
        
        for i in range(len(meter_scaled) - self.sequence_length):
            X_meter.append(meter_scaled[i:(i + self.sequence_length)])
            X_weather.append(weather_scaled[i + self.sequence_length - 1])  # Current weather
            X_tariff.append(tariff_scaled[i + self.sequence_length - 1])    # Current tariff
            y.append(target_scaled[i + self.sequence_length])
        
        return [np.array(X_meter), np.array(X_weather), np.array(X_tariff)], np.array(y)

In [ ]:
class ModelBenchmark:
    def __init__(self):
        self.models = {}
        self.results = {}
    
    def create_baseline_models(self, input_shape):
        seq_len, n_features = input_shape
        
        # LSTM baseline
        lstm_model = keras.Sequential([
            layers.LSTM(128, return_sequences=True, input_shape=(seq_len, n_features)),
            layers.LSTM(64),
            layers.Dense(32, activation='relu'),
            layers.Dense(1)
        ])
        
        # CNN baseline  
        cnn_model = keras.Sequential([
            layers.Conv1D(64, 3, activation='relu', input_shape=(seq_len, n_features)),
            layers.Conv1D(32, 3, activation='relu'),
            layers.GlobalMaxPooling1D(),
            layers.Dense(50, activation='relu'),
            layers.Dense(1)
        ])
        
        # GRU baseline
        gru_model = keras.Sequential([
            layers.GRU(128, return_sequences=True, input_shape=(seq_len, n_features)),
            layers.GRU(64),
            layers.Dense(32, activation='relu'),
            layers.Dense(1)
        ])
        
        return {'LSTM': lstm_model, 'CNN': cnn_model, 'GRU': gru_model}
    
    def train_baseline_models(self, models, X_train, y_train, X_val, y_val):
        for name, model in models.items():
            print(f"Training {name}...")
            model.compile(optimizer='adam', loss='mse', metrics=['mae'])
            
            history = model.fit(
                X_train, y_train,
                validation_data=(X_val, y_val),
                epochs=50,
                batch_size=32,
                verbose=0,
                callbacks=[keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)]
            )
            
            self.models[name] = model
    
    def evaluate_all_models(self, X_test, y_test, target_scaler):
        for name, model in self.models.items():
            if name == 'XGBoost':
                # Handle XGBoost separately
                X_test_flat = X_test.reshape(X_test.shape[0], -1)
                y_pred = model.predict(X_test_flat)
            else:
                y_pred = model.predict(X_test)
            
            y_pred_original = target_scaler.inverse_transform(y_pred.reshape(-1, 1)).flatten()
            y_test_original = target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
            
            rmse = np.sqrt(mean_squared_error(y_test_original, y_pred_original))
            mae = mean_absolute_error(y_test_original, y_pred_original)
            r2 = r2_score(y_test_original, y_pred_original)
            
            self.results[name] = {'RMSE': rmse, 'MAE': mae, 'R2': r2}
        
        return self.results
    
    def add_xgboost_model(self, X_train, y_train, X_val, y_val):
        X_train_flat = X_train.reshape(X_train.shape[0], -1)
        X_val_flat = X_val.reshape(X_val.shape[0], -1)
        
        xgb_model = xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.1,
            early_stopping_rounds=10,
            random_state=42
        )

        # Fit the model with evaluation sets
        xgb_model.fit(
            X_train_flat, y_train,
            eval_set=[(X_train_flat, y_train), (X_val_flat, y_val)],
            verbose=True
        )

        
        self.models['XGBoost'] = xgb_model
    
    def plot_comparison(self):
        metrics = ['RMSE', 'MAE', 'R2']
        models = list(self.results.keys())
        
        fig, axes = plt.subplots(1, 3, figsize=(15, 5))
        
        for i, metric in enumerate(metrics):
            values = [self.results[model][metric] for model in models]
            axes[i].bar(models, values)
            axes[i].set_title(f'{metric} Comparison')
            axes[i].set_ylabel(metric)
            plt.setp(axes[i].xaxis.get_majorticklabels(), rotation=45)
        
        plt.tight_layout()
        plt.show()

In [ ]:
def train_enhanced_granx(X_train, y_train, X_val, y_val, epochs=100):
    # Extract shapes
    meter_shape = X_train[0].shape
    weather_shape = X_train[1].shape[1]
    tariff_shape = X_train[2].shape[1]
    
    model = EnhancedGRANXModel(
        sequence_length=meter_shape[1],
        n_features=meter_shape[2],
        weather_features=weather_shape,
        tariff_features=tariff_shape,
        hidden_units=128,
        attention_heads=8
    )
    
    # Build model by running a forward pass
    dummy_batch = [x[:1] for x in X_train]  # Take first sample from each input
    _ = model(dummy_batch, training=False)
    
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='mse',
        metrics=['mae']
    )
    
    callbacks = [
        keras.callbacks.EarlyStopping(patience=15, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=8, factor=0.5, min_lr=1e-6)
    ]
    
    history = model.fit(
        X_train, y_train,
        validation_data=(X_val, y_val),
        epochs=epochs,
        batch_size=32,
        callbacks=callbacks,
        verbose=1
    )
    
    return model, history

In [ ]:
def comprehensive_evaluation():
    print("Enhanced GRAN-X Comprehensive Evaluation")
    print("="*60)
    
    # Create heterogeneous data
    processor = HeterogeneousDataProcessor(sequence_length=24)
    meter_data, weather_data, tariff_data = processor.create_heterogeneous_synthetic_data(n_samples=3000)
    
    # Create sequences
    X, y = processor.create_sequences_heterogeneous(meter_data, weather_data, tariff_data)
    
    # Split data
    split_train = int(0.7 * len(y))
    split_val = int(0.85 * len(y))
    
    X_train = [x[:split_train] for x in X]
    X_val = [x[split_train:split_val] for x in X]
    X_test = [x[split_val:] for x in X]
    
    y_train, y_val, y_test = y[:split_train], y[split_train:split_val], y[split_val:]
    
    print(f"Training samples: {len(y_train)}")
    print(f"Validation samples: {len(y_val)}")  
    print(f"Test samples: {len(y_test)}")
    
    # Train Enhanced GRAN-X
    print("\nTraining Enhanced GRAN-X...")
    granx_model, history = train_enhanced_granx(X_train, y_train, X_val, y_val, epochs=60)
    
    # Benchmark against baselines
    print("\nTraining baseline models...")
    benchmark = ModelBenchmark()
    
    # Use only meter data for baseline comparison
    baseline_models = benchmark.create_baseline_models(X_train[0].shape[1:])
    benchmark.train_baseline_models(baseline_models, X_train[0], y_train, X_val[0], y_val)
    
    # Add XGBoost
    benchmark.add_xgboost_model(X_train[0], y_train, X_val[0], y_val)
    
    # Add GRAN-X results
    y_pred_granx = granx_model.predict(X_test)
    y_test_orig = processor.target_scaler.inverse_transform(y_test.reshape(-1, 1)).flatten()
    y_pred_orig = processor.target_scaler.inverse_transform(y_pred_granx.reshape(-1, 1)).flatten()
    
    benchmark.results['GRAN-X'] = {
        'RMSE': np.sqrt(mean_squared_error(y_test_orig, y_pred_orig)),
        'MAE': mean_absolute_error(y_test_orig, y_pred_orig),
        'R2': r2_score(y_test_orig, y_pred_orig)
    }
    
    # Evaluate all models
    baseline_results = benchmark.evaluate_all_models(X_test[0], y_test, processor.target_scaler)
    
    # Print results
    print("\nModel Performance Comparison:")
    print("-" * 50)
    for model, metrics in benchmark.results.items():
        print(f"{model:12} | RMSE: {metrics['RMSE']:6.2f} | MAE: {metrics['MAE']:6.2f} | R²: {metrics['R2']:6.4f}")
    
    # Plot comparison
    benchmark.plot_comparison()
    
    return granx_model, benchmark.results, processor

In [ ]:
if __name__ == "__main__":
    model, results, processor = comprehensive_evaluation()